In [1]:
import sys
import boto3
import pandas as pd
import sagemaker

from sagemaker.train import ModelTrainer

from sagemaker.train.configs import (
    SourceCode,
    Compute,
    InputData,
    S3DataSource,
    OutputDataConfig,
    StoppingCondition
)

print("Imports successful")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Imports successful


In [2]:
session = boto3.Session()

region = session.region_name

print("AWS Region:")
print(region)

AWS Region:
ap-south-1


In [3]:
print("Python executable:")
print(sys.executable)

print()

print("Python version:")
print(sys.version)

print()

print("SageMaker module:")
print(sagemaker.__file__)

Python executable:
/opt/conda/bin/python

Python version:
3.12.13 | packaged by conda-forge | (main, Aug 11 2026, 10:27:28) [GCC 14.4.0]

SageMaker module:
/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py


In [4]:
role = "arn:aws:iam::432649419164:role/sagemaker-full"

print("SageMaker execution role:")
print(role)

SageMaker execution role:
arn:aws:iam::432649419164:role/sagemaker-full


In [5]:
bucket = "sagemaker-ap-south-1-432649419164"

train_s3_uri = f"s3://{bucket}/input/train.csv"

train_s3_prefix = f"s3://{bucket}/input/"

output_s3_uri = f"s3://{bucket}/output/"

print("Training file:")
print(train_s3_uri)

print()

print("Training prefix:")
print(train_s3_prefix)

print()

print("Output location:")
print(output_s3_uri)

Training file:
s3://sagemaker-ap-south-1-432649419164/input/train.csv

Training prefix:
s3://sagemaker-ap-south-1-432649419164/input/

Output location:
s3://sagemaker-ap-south-1-432649419164/output/


In [6]:
df = pd.read_csv(train_s3_uri)

print("Dataset shape:")
print(df.shape)

print()

print("Dataset:")
print(df)

Dataset shape:
(16, 4)

Dataset:
    area_sqft  bedrooms  age_years   price
0        1200         2         10  250000
1        1500         3          5  320000
2        1800         3          8  380000
3        2000         4          4  450000
4        1000         2         15  210000
5        2200         4          3  500000
6        1300         2         12  270000
7        1700         3          7  360000
8        1400         3         10  300000
9        1900         4          6  420000
10       1100         2         20  220000
11       2100         4          2  480000
12       1600         3          9  340000
13       1250         2         11  260000
14       1750         3          6  370000
15       2300         4          5  510000


In [7]:
source_code = SourceCode(
    source_dir=".",
    entry_script="train2.py"
)

print("Source code configured")

Source code configured


In [8]:
compute = Compute(
    instance_type="ml.m5.large",
    instance_count=1
)

print("Compute configured")

Compute configured


In [9]:
s3_data_source = S3DataSource(
    s3_data_type="S3Prefix",
    s3_uri=train_s3_prefix
)

In [10]:
input_data = InputData(
    channel_name="train",
    data_source=s3_data_source,
    content_type="text/csv"
)

print("Training input configured")

Training input configured


In [11]:
output_data = OutputDataConfig(
    s3_output_path=output_s3_uri
)

print("Output configuration created")

Output configuration created


In [12]:
stopping_condition = StoppingCondition(
    max_runtime_in_seconds=600
)

print("Maximum runtime: 600 seconds")

Maximum runtime: 600 seconds


In [13]:
from sagemaker.core import image_uris
training_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.4-2",
    py_version="py3",
    instance_type="ml.m5.large"
)

print("Training image:")
print(training_image)

Training image:
720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-learn:1.4-2-cpu-py3


In [14]:
trainer = ModelTrainer(
    role=role,
    base_job_name="simple-ml-training",
    source_code=source_code,
    compute=compute,
    training_image=training_image,
    output_data_config=output_data,
    input_data_config=[
        input_data
    ],
    stopping_condition=stopping_condition,
    training_input_mode="File"
)

print("ModelTrainer created successfully")

[08/30/26 03:38:32] INFO     SageMaker session not provided. Using default Session.                  ]8;id=9094376;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=9094377;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#61\61]8;;\

                    INFO     OutputDataConfig compression type not provided. Using default:         ]8;id=9094383;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=9094384;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#165\165]8;;\
                             GZIP                                                                                  

                    INFO     Training image URI:                                               ]8;id=9094391;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=9094392;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             720646828776.dkr.ecr.ap-south-1.amazonaws.com/sagemaker-scikit-le                     
                             arn:1.4-2-cpu-py3                                                                     

ModelTrainer created successfully


In [15]:
training_job = trainer.train()

print("Training job:")
print(training_job)

[08/30/26 03:38:42] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=9094399;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=9094400;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#304\304]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


[08/30/26 03:38:43] INFO     Creating training_job resource.                                     ]8;id=9094407;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9094408;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31238\31238]8;;\

                    WARNING  No region provided. Using default region.                                 ]8;id=9094415;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=9094416;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/utils/utils.py#361\361]8;;\

Output()

[08/30/26 03:40:35] INFO     simple-ml-training-20260830033842/algo-1-1788061169:                ]8;id=9094422;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9094423;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             Starting training script                                                              

... (123 outputs left out)

                    INFO     simple-ml-training-20260830033842/algo-1-1788061169:                ]8;id=9095178;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9095179;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             Training Container Execution Completed                                                

                    INFO     simple-ml-training-20260830033842/algo-1-1788061169:                ]8;id=9095184;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9095185;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31584\31584]8;;\
                             ++ echo 'Training Container Execution Completed'                                      

[08/30/26 03:40:45] INFO     Final Resource Status: Completed                                    ]8;id=9095191;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=9095192;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/resources.py#31590\31590]8;;\

Training job:
None


In [21]:
import boto3

s3 = boto3.client("s3")

bucket = "sagemaker-ap-south-1-432649419164"

model_key = "output/simple-ml-training-20260830033140/output/model.tar.gz"

s3.download_file(
    bucket,
    model_key,
    "/tmp/model.tar.gz"
)

print("Downloaded model artifact")

Downloaded model artifact


In [22]:
import tarfile
import os

with tarfile.open("/tmp/model.tar.gz", "r:gz") as tar:
    tar.extractall("/tmp/model")

print(os.listdir("/tmp/model"))

['model.joblib']


/tmp/ipykernel_2064/4184483.py:5: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("/tmp/model")


In [24]:
import joblib


model = joblib.load("/tmp/model/model.joblib")

prediction = model.predict(
    pd.DataFrame({
        "area_sqft": [1600],
        "bedrooms": [3],
        "age_years": [5]
    })
)

print(prediction)

[347324.64428471]
